# Quantum reference frames: the \(Z_2\) example

**Working notes by Ashesh Kumar Gupta**

This notebook follows the three-qubit example in Castro-Ruiz, Galley and Loveridge, *Interpreting quantum-reference-frame transformations through a simple example*, Phys. Rev. A **114**, 032201 (2026). I first reproduce the main calculations, then look at a few extensions.

I distinguish calculations taken from that paper from my own checks and extensions. The \(U(1)\) non-ideal-frame example in Section 8 comes from a separate preprint by Garmier, Hausmann and Castro-Ruiz (arXiv:2512.19343). 

**The extensions in Sections 7, 9, 10 and 11 are exploratory and need more brainstroming.**

The main questions are:

1. Why do the two global states give the same state relative to Alice but orthogonal states relative to Bob?
2. Which observables can distinguish the states in each description?
3. What changes when the reference frame is non-ideal?
4. Can the distinguishability of different charge sectors be described directly in terms of the \(\kappa\)-sets?

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

I2 = np.eye(2); X = np.array([[0,1],[1,0]], dtype=complex)
def kron(*ops):
    out = np.array([[1]], dtype=complex)
    for o in ops: out = np.kron(out, o)
    return out
def ket(bits):
    v = np.zeros(8, dtype=complex); v[int(bits,2)] = 1; return v
def ket2(bits):
    v = np.zeros(4, dtype=complex); v[int(bits,2)] = 1; return v

G = kron(X,X,X)                       # global Z2 generator X_A (x) X_B (x) X_C
Pi0 = (np.eye(8)+G)/2                 # projector on trivial sector H0
Pi1 = (np.eye(8)-G)/2                 # projector on sign sector H1

# Eqs. (12), (13)
psi0 = 0.5*((ket('001')+ket('110')) - (ket('100')+ket('011')))
psi1 = 0.5*((ket('001')-ket('110')) + (ket('100')-ket('011')))
print("norms:", np.linalg.norm(psi0), np.linalg.norm(psi1))
print("psi0 in H0:", np.allclose(G@psi0, psi0), "| psi1 in H1:", np.allclose(G@psi1, -psi1))

norms: 1.0 1.0
psi0 in H0: True | psi1 in H1: True


## 1. Reduction maps

The reduction map for Alice is
$
R_{0|A}|\psi\rangle=\sqrt{2}\,{}_A\langle0|\psi\rangle.
$
It keeps the terms with Alice's qubit in \(|0\rangle\), with the factor \(\sqrt{2}\) ensuring normalisation in this example.

For
$$
|\psi_0\rangle=\tfrac12(|001\rangle+|110\rangle-|100\rangle-|011\rangle),
$$
$$
|\psi_1\rangle=\tfrac12(|001\rangle-|110\rangle+|100\rangle-|011\rangle),
$$
the \(A=0\) terms are the same. Thus both states give
$$
|\psi_0\rangle^A=|\psi_1\rangle^A
=\frac{|01\rangle-|11\rangle}{\sqrt2}.
$$

For Bob, the \(B=0\) terms differ by a sign:
$$
|\psi_0\rangle^B=\frac{|01\rangle-|10\rangle}{\sqrt2},
\qquad
|\psi_1\rangle^B=\frac{|01\rangle+|10\rangle}{\sqrt2}.$$
These two states are orthogonal. The next cell checks both reductions.

In [2]:
def reduce_frame(psi, frame):
    """R_{0|frame}: sqrt(2) <0|_frame psi. Remaining qubits keep their order."""
    t = psi.reshape(2,2,2)
    out = np.sqrt(2)*np.take(t, 0, axis=frame)
    return out.reshape(4)

psiA0, psiA1 = reduce_frame(psi0,0), reduce_frame(psi1,0)   # BC relative to A
psiB0, psiB1 = reduce_frame(psi0,1), reduce_frame(psi1,1)   # AC relative to B

print("BC | A :  psi0 ->", psiA0, "\n          psi1 ->", psiA1)
print("same relative state for Alice:", np.allclose(psiA0, psiA1))
print("\nAC | B :  psi0 ->", psiB0, "\n          psi1 ->", psiB1)
print("overlap <psiB0|psiB1> =", np.vdot(psiB0, psiB1), "(orthogonal)")

BC | A :  psi0 -> [ 0.    +0.j  0.7071+0.j  0.    +0.j -0.7071+0.j] 
          psi1 -> [ 0.    +0.j  0.7071+0.j  0.    +0.j -0.7071+0.j]
same relative state for Alice: True

AC | B :  psi0 -> [ 0.    +0.j  0.7071+0.j -0.7071+0.j  0.    +0.j] 
          psi1 -> [0.    +0.j 0.7071+0.j 0.7071+0.j 0.    +0.j]
overlap <psiB0|psiB1> = 0j (orthogonal)


## 2. Relative observables

For an observable \(T\) acting on \(BC\), the observable relative to Alice is
$
\Phi^A_{BC}(T)=|0\rangle\langle0|\otimes T+
|1\rangle\langle1|\otimes(X\otimes X)T(X\otimes X).
$

To see why this gives the expected relative statistics, write
$(P=|0\rangle\langle0|\otimes T)$ and
$(G=X\otimes X\otimes X)$. Then
$$
\Phi^A_{BC}(T)=P+GPG^\dagger.
$$
For either global state, $(G|\psi\rangle=\pm|\psi\rangle)$. The sign cancels in the expectation value, giving
$$
\langle\psi|\Phi^A_{BC}(T)|\psi\rangle
=2\langle\psi|P|\psi\rangle
=\langle\psi^A|T|\psi^A\rangle.
$$
Since the two Alice-relative states are equal, no observable relative to Alice distinguishes them. The corresponding construction for Bob gives the same relation, but Bob's relative states are orthogonal, so an appropriate relative observable can distinguish them.

In [3]:
rng = np.random.default_rng(0)
P0 = np.diag([1,0]).astype(complex); P1 = np.diag([0,1]).astype(complex)
XX = np.kron(X,X)

def rel_obs_A(T):   # T acts on B(x)C
    return np.kron(P0,T) + np.kron(P1, XX@T@XX)

def embed_AC(T):
    """T on A(x)C  ->  T (x) I_B on ordering A,B,C."""
    T4 = T.reshape(2,2,2,2)                       # a c ; a' c'
    full = np.zeros((2,2,2,2,2,2), dtype=complex)  # a b c ; a' b' c'
    for b in range(2):
        full[:,b,:,:,b,:] = T4.reshape(2,2,2,2)
    return full.reshape(8,8)

def rel_obs_B(T):
    out = np.zeros((8,8), dtype=complex)
    for j,Pj in enumerate([P0,P1]):
        U = kron(X if j else I2, I2, X if j else I2)
        Pjb = kron(I2, Pj, I2)
        out += Pjb @ (U @ embed_AC(T) @ U.conj().T)
    return out

def rand_herm(n):
    M = rng.normal(size=(n,n))+1j*rng.normal(size=(n,n)); return (M+M.conj().T)/2

def ev(psi,O): return np.vdot(psi, O@psi).real

# sanity: relative observables are invariant under the global action
T = rand_herm(4)
print("Phi^A invariant:", np.allclose(G@rel_obs_A(T)@G, rel_obs_A(T)),
      "| Phi^B invariant:", np.allclose(G@rel_obs_B(T)@G, rel_obs_B(T)))

dA = max(abs(ev(psi0,rel_obs_A(T))-ev(psi1,rel_obs_A(T))) for T in [rand_herm(4) for _ in range(200)])
dB = max(abs(ev(psi0,rel_obs_B(T))-ev(psi1,rel_obs_B(T))) for T in [rand_herm(4) for _ in range(200)])
print(f"max |<O>_psi0 - <O>_psi1|  over 200 random observables\n   relative to A: {dA:.2e}\n   relative to B: {dB:.3f}")

# eqs. (A3)/(A5): evaluating relative observable on global state = local observable on relative state
T = rand_herm(4)
print("\nEq. (A3):", np.isclose(ev(psi0,rel_obs_A(T)), np.vdot(psiA0,T@psiA0).real))
print("Eq. (A5):", np.isclose(ev(psi0,rel_obs_B(T)), np.vdot(psiB0,T@psiB0).real))

Phi^A invariant: True | Phi^B invariant: True
max |<O>_psi0 - <O>_psi1|  over 200 random observables
   relative to A: 0.00e+00
   relative to B: 3.941

Eq. (A3): True
Eq. (A5): True


## 3. The global charge observable

The two states belong to different eigenspaces of
$
G=X_A\otimes X_B\otimes X_C.
$
In particular,
$$
G|\psi_0\rangle=+|\psi_0\rangle,\qquad
G|\psi_1\rangle=-|\psi_1\rangle.
$$
The expectation values are therefore \(+1\) and \(-1\). Although \(G\) is invariant under the group action, it is not an observable of the form $(\Phi^A_{BC}(T))$. It carries information about the global charge that is absent from Alice's relative description.

In [4]:
print("<X(x)X(x)X>  psi0:", ev(psi0,G), "  psi1:", ev(psi1,G))
print("-> invariant observable that distinguishes the two global states, but is NOT a relative observable.")

<X(x)X(x)X>  psi0: 1.0   psi1: -1.0
-> invariant observable that distinguishes the two global states, but is NOT a relative observable.


## 4. The operational description

For Bob's relative states, consider framed observables of the form
$[
|i\rangle\langle i|_A\otimes T_C.
]$
For the singlet-like state
$$
|\psi_0\rangle^B=(|01\rangle-|10\rangle)/\sqrt2,
$$
the two diagonal blocks give equal-weight contributions from \(C=1\) and \(C=0\). The relative sign between the branches does not affect these statistics. The same framed observables therefore give identical statistics for $(|\psi_0\rangle^B)$ and $(|\psi_1\rangle^B)$.

For equal prior probabilities, the corresponding mixture is
$$
\bar\rho=\frac12\bigl(|01\rangle\langle01|+
|10\rangle\langle10|\bigr)$$
as in Eq. (31) of the paper. The code checks this equality for a collection of framed observables.

In [5]:
# Bob-relative states on A(x)C
rho0 = np.outer(psiB0, psiB0.conj()); rho1 = np.outer(psiB1, psiB1.conj())
rho_bar = 0.5*(rho0+rho1)

# observables |i><i|_A (x) T_C, with random Hermitian T_C
def framed(i):
    Pi = [P0,P1][i]
    return lambda Tc: np.kron(Pi, Tc)
gap = max(abs(np.trace(rho0@framed(i)(Tc)).real - np.trace(rho1@framed(i)(Tc)).real)
          for i in (0,1) for Tc in [rand_herm(2) for _ in range(200)])
print(f"max difference on framed observables |i><i|_A (x) T_C: {gap:.2e}  (indistinguishable by Alice)")

target = 0.5*(np.outer(ket2('01'),ket2('01'))+np.outer(ket2('10'),ket2('10')))
print("rho_bar = (|01><01| + |10><10|)/2 :", np.allclose(rho_bar, target))
print("rho_bar gives the same framed statistics:",
      all(np.isclose(np.trace(rho_bar@framed(i)(Tc)), np.trace(rho0@framed(i)(Tc))) for i in (0,1) for Tc in [rand_herm(2) for _ in range(20)]))

max difference on framed observables |i><i|_A (x) T_C: 0.00e+00  (indistinguishable by Alice)
rho_bar = (|01><01| + |10><10|)/2 : True
rho_bar gives the same framed statistics: True


## 5. Changing the quantum reference frame

### 5.1 Extra-particle construction

Here I reproduce the output of the frame change for the two states, rather than implementing Eq. (26) directly.

Within a fixed charge sector $(s=\pm1)$, the map $(R_{0|A})$ can be inverted:
$$
|\phi\rangle\longmapsto
\sqrt2\,\Pi_s\bigl(|0\rangle_A\otimes|\phi\rangle\bigr),
\qquad
\Pi_s=(1+sG)/2.
$$
I reconstruct the global state in this way and then apply $(R_{0|B})$. This gives the same two Alice-to-Bob state transformations as the example in the paper.

The code does not construct the controlled unitaries $(U_\pm)$ or the map $(S_{A\to B})$ itself, and it only checks the two states used in the example. A direct implementation would be a separate task.

### 5.2 Operational construction

The next calculation implements \(Q\) from Eq. (B12) and checks its action on $(\bar\rho)$. It is not a general test of invertibility.

In [6]:
# --- Extra-particle approach, Eqs. (24)-(26): S_{A->B} on the (A,B) pair of the BC-relative description
# Implemented via the global-state route: invert R_{0|A} within the charge sector, then apply R_{0|B}.
def to_bob_from_global(psi):  return reduce_frame(psi,1)
def to_alice_from_global(psi): return reduce_frame(psi,0)

def extra_particle_change(psiA, charge):
    """Invert R_{0|A} on the charge sector, then apply R_{0|B}."""
    # global vector from Alice's relative state: sqrt2 * Pi_charge (|0>_A (x) psiA)
    glob = np.sqrt(2)*np.kron(np.array([1,0],dtype=complex), psiA)
    Pi = Pi0 if charge>0 else Pi1
    glob = Pi@glob
    glob = glob/np.linalg.norm(glob)
    return reduce_frame(glob,1)

print("charge +  :", np.allclose(extra_particle_change(psiA0,+1), psiB0), " (-> psi0 relative to B)")
print("charge -  :", np.allclose(extra_particle_change(psiA1,-1), psiB1), " (-> psi1 relative to B)")
print("The same Alice state is mapped to different Bob states depending on the charge,")
print("so the map is only well-defined once the 'extra particle' information is supplied.\n")

# --- Operational invertible map Q (Eq. B12) on framed, dephased states
def dephase_B(rho):      # D: rho_BC -> sum_k |k><k|_B (x) <k|rho|k>
    out = np.zeros_like(rho)
    for k,Pk in enumerate([P0,P1]):
        K = np.kron(Pk, I2); out += K@rho@K
    return out
def Q(rho_BC):           # Eq. (B12): sum_i |i><i|_A (x) <i|_B X^i_C rho X^i_C |i>_B  (A,C ordering)
    out = np.zeros((4,4), dtype=complex)
    for i in (0,1):
        Xc = np.kron(I2, X if i else I2)
        # <i|_B on (B,C): picks block
        blk = (Xc@rho_BC@Xc)[2*i:2*i+2, 2*i:2*i+2]    # C-block conditioned on B=i
        out += np.kron([P0,P1][i], blk)
    return out
rhoA0 = np.outer(psiA0, psiA0.conj())
rbar_A = dephase_B(rhoA0)
print("Q maps rho_bar(A) to rho_bar(B):", np.allclose(Q(rbar_A), rho_bar))

charge +  : True  (-> psi0 relative to B)
charge -  : True  (-> psi1 relative to B)
The same Alice state is mapped to different Bob states depending on the charge,
so the map is only well-defined once the 'extra particle' information is supplied.

Q maps rho_bar(A) to rho_bar(B): True


## 6. Extending the example to \(Z_N\)

I next replace the $(Z_2)$ shift by a cyclic shift of order \(N\). Let
$$S|j\rangle=|j+1\rangle,\qquad
\omega=e^{2\pi i/N},
$$
and take the global action $(U=S\otimes S\otimes S)$. Starting from an arbitrary Alice-relative state $(|\phi\rangle_{BC})$, define
$$
|\Psi_q\rangle=\frac1{\sqrt N}\sum_{g=0}^{N-1}
\omega^{-qg}|g\rangle_A\otimes
(S^g\otimes S^g)|\phi\rangle_{BC}.
$$
With this convention, $(U|\Psi_q\rangle=\omega^q|\Psi_q\rangle)$. Also,
$
R_{0|A}|\Psi_q\rangle=|\phi\rangle
$
for every \(q\). Thus the Alice-relative state does not depend on the global charge label.

For Bob, taking the \(B=0\) component gives states related by
$
|\Psi_q\rangle^B=(Z_A^{-q}\otimes I_C)|\Psi_0\rangle^B,
$
where $(Z=\operatorname{diag}(1,\omega,\ldots,\omega^{N-1}))$. Their overlaps are
$
\langle\Psi_p^B|\Psi_q^B\rangle
=\operatorname{Tr}(\rho_A Z^{p-q}),
$
with $(\rho_A)$ the reduced state of $(|\Psi_0\rangle^B)$ on \(A\).

The \(N\) Bob-relative states are mutually orthogonal precisely when the diagonal entries of $(\rho_A)$ are uniform. The $(Z_2)$ example in the paper is a special case. For a general starting state $(|\phi\rangle)$, the states need not be orthogonal.

In [8]:
rng = np.random.default_rng(1)
def zn_bob_states(N, phi):
    w = np.exp(2j*np.pi/N); out = []
    for q in range(N):
        psi = np.zeros((N,N,N), dtype=complex)                      # indices a,b,c
        for g in range(N):
            psi[g] += w**(-q*g) * np.roll(np.roll(phi,g,axis=0),g,axis=1) / np.sqrt(N)
        assert np.allclose(np.sqrt(N)*psi[0], phi)                  # same Alice state for all q
        out.append((psi, np.sqrt(N)*psi[:,0,:]))                    # global state, Bob state <0|_B
    return out

for N in (2,3,4,5):
    phi = rng.normal(size=(N,N)) + 1j*rng.normal(size=(N,N)); phi /= np.linalg.norm(phi)
    st = zn_bob_states(N, phi); Bob = [b for _,b in st]
    Z = lambda k: np.diag(np.exp(2j*np.pi*k*np.arange(N)/N))
    twist = all(np.allclose(Bob[q], Z(-q)@Bob[0]) for q in range(N))
    # charge eigenvalue check: U psi_q = w^q psi_q
    S = np.roll(np.eye(N),1,axis=0)
    ok_charge = all(np.allclose(np.einsum('ai,bj,ck,ijk->abc',S,S,S,st[q][0]), np.exp(2j*np.pi*q/N)*st[q][0]) for q in range(N))
    rhoA = Bob[0]@Bob[0].conj().T                                   # Bob_0 on (A,C) -> reduce to A
    ov_formula = all(np.isclose(np.vdot(Bob[p],Bob[q]), np.trace(rhoA@Z(p-q))) for p in range(N) for q in range(N))
    print(f"Z_{N}: twist {twist} | charge eigenvalue w^q {ok_charge} | overlap = Tr[rho_A Z^(p-q)] {ov_formula}")

# engineered case: uniform diagonal of rho_A  ->  mutually orthogonal Bob states (the Z_2 paper case generalised)
N=4
v = np.zeros((N,N),dtype=complex)                                   # Bob_0[a,c] = delta_{c, a}/sqrt(N): uniform on A
for a in range(N): v[a,a] = 1/np.sqrt(N)
# build phi from Bob_0: Bob_0[a,c] = phi(-a, c-a)  ->  phi[b,c'] with b=-a, c'=c-a
phi = np.zeros((N,N),dtype=complex)
for a in range(N):
    for c in range(N): phi[(-a)%N,(c-a)%N] = v[a,c]
Bob = [b for _,b in zn_bob_states(N, phi)]
G = np.array([[abs(np.vdot(Bob[p],Bob[q])) for q in range(N)] for p in range(N)])
print(); print("uniform-diagonal case, |overlap| matrix:"); print(G)

Z_2: twist True | charge eigenvalue w^q True | overlap = Tr[rho_A Z^(p-q)] True
Z_3: twist True | charge eigenvalue w^q True | overlap = Tr[rho_A Z^(p-q)] True
Z_4: twist True | charge eigenvalue w^q True | overlap = Tr[rho_A Z^(p-q)] True
Z_5: twist True | charge eigenvalue w^q True | overlap = Tr[rho_A Z^(p-q)] True

uniform-diagonal case, |overlap| matrix:
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


## 7. A non-ideal \(U(1)\) frame

This section reproduces Example 5 of Garmier, Hausmann and Castro-Ruiz, *The Perspectives of Non-Ideal Quantum Reference Frames* (arXiv:2512.19343).

I use the orientation POVM
$$
\gamma_F(\theta)=\frac{|\theta\rangle\langle\theta|}{2\pi},
\qquad
|\theta\rangle=\sum_{r\in\sigma_F}e^{ir\theta}|r\rangle,
$$
and the relationalisation map
$$
R_F(f)=\int d\theta\,
\gamma_F(\theta)\otimes U_S(\theta)fU_S(\theta)^\dagger.
$$
For the finite spectra used here, the integrand is a finite trigonometric polynomial, so the integral can be evaluated on a sufficiently fine uniform grid.

The preprint organises the Hilbert space into sectors labelled by
$$
\kappa(r)=\sigma_S\cap(r-\sigma_F).
$$
The dimension of the generated algebra is obtained by summing $(|\kappa|^2)$ over the distinct non-empty sectors. For the two frame/system choices below, this gives dimensions 6 and 7. The code checks these values by explicitly generating the algebra from the relational operators.

I also vary a relative phase in the frame state. The phase is invisible to individual relational operators in this example, but it can appear in products of those operators. The comparison below is restricted to Hermitian operators.

In [9]:
K=32; thetas=2*np.pi*np.arange(K)/K; wq=2*np.pi/K
def frame_povm(spec):
    spec=np.array(spec)
    return [np.outer(np.exp(1j*spec*t), np.exp(-1j*spec*t))/(2*np.pi) for t in thetas]  # |theta><theta|/2pi
def U(spec,t): return np.diag(np.exp(1j*np.array(spec)*t))
def relational_ops(specF, specS):
    '''R_F(f) for f = matrix units of S. Ordering: F (x) S.'''
    gam=frame_povm(specF); dS=len(specS); ops=[]
    for i in range(dS):
        for j in range(dS):
            E=np.zeros((dS,dS),dtype=complex); E[i,j]=1
            R=sum(wq*np.kron(gam[k], U(specS,t)@E@U(specS,t).conj().T) for k,t in enumerate(thetas))
            ops.append(R)
    return ops
def span_basis(mats,tol=1e-9):
    M=np.array([m.flatten() for m in mats]); u,s,vh=np.linalg.svd(M,full_matrices=False)
    r=(s>tol).sum(); return [vh[i].reshape(mats[0].shape) for i in range(r)]
def generated_algebra_dim(ops):
    d=ops[0].shape[0]; B=span_basis(ops+[np.eye(d)])
    while True:
        new=span_basis(B+[a@b for a in B for b in B]+[a.conj().T for a in B])
        if len(new)==len(B): return len(B)
        B=new
sA=[-1,0,1]; sB=[-1,1]
opsA=relational_ops(sA,sB)           # frame A=C^3, system B=C^2
opsB=relational_ops(sB,sA)           # frame B=C^2, system A=C^3
print("dim of R_A[O_B] image:", len(span_basis(opsA)), "| generated algebra dim:", generated_algebra_dim(opsA), "(paper: 6 > 4)")
print("dim of R_B[O_A] image:", len(span_basis(opsB)), "| generated algebra dim:", generated_algebra_dim(opsB), "(paper: 7 < 9)")
# regular-frame check: ideal frame sigma_A = Z_3-like large -> algebra dim should equal dim O_B = 4

# hidden relative phase phi: invisible to relational operators, visible to products
sig=np.diag([0.5,0.5])
def state(phi):
    th0,th1=0.0,2*np.pi/3
    ket=lambda t: np.exp(1j*np.array(sA)*t)
    v=(ket(th0)+np.exp(1j*phi)*ket(th1))/np.sqrt(2)
    v=v/np.linalg.norm(v)
    return np.kron(np.outer(v,v.conj()),sig)
phis=np.linspace(0,2*np.pi,7)
def herm_parts(a,b):                      # Hermitian combinations of a product: (ab+ba)/2 and i(ab-ba)/2
    return [(a@b+b@a)/2, 1j*(a@b-b@a)/2]
H1=[o for o in opsA if np.allclose(o,o.conj().T)]                 # Hermitian single relational operators (matrix units give mostly non-Hermitian ones)
H1+= [ (o+o.conj().T)/2 for o in opsA]+[1j*(o-o.conj().T)/2 for o in opsA]
spread1=max(abs(np.trace(state(p)@o)-np.trace(state(0)@o)) for p in phis for o in H1)
prodH=[h for a in H1 for b in H1 for h in herm_parts(a,b)]
spread2=max(abs(np.trace(state(p)@o)-np.trace(state(0)@o)) for p in phis for o in prodH)
print("phi-dependence over Hermitian observables:"); print(f"  single relational operators : {spread1:.2e}"); print(f"  Hermitian parts of products : {spread2:.3f}")

# analytic sector counting (paper's kappa map): algebra dim = sum over distinct sectors k of |k|^2
def kappa_dim(sigF, sigS):
    sigAS = sorted({a+b for a in sigF for b in sigS})
    sectors = {frozenset(set(sigS) & {r-a for a in sigF}) for r in sigAS}
    return sum(len(k)**2 for k in sectors if len(k)>0), sectors
print(); print("kappa counting  A frame (system B):", kappa_dim(sA,sB)[0], "  B frame (system A):", kappa_dim(sB,sA)[0])

dim of R_A[O_B] image: 4 | generated algebra dim: 6 (paper: 6 > 4)
dim of R_B[O_A] image: 5 | generated algebra dim: 7 (paper: 7 < 9)
phi-dependence over Hermitian observables:
  single relational operators : 1.11e-16
  Hermitian parts of products : 0.083

kappa counting  A frame (system B): 6   B frame (system A): 7


## 8. Bayesian inference in the \(Z_2\) example

Suppose Alice assigns prior probabilities $(p_0,p_1)$ to the two charge sectors. This is classical uncertainty about which state was prepared, rather than a coherent superposition of the two states.

Because the Alice-relative states are identical, every measurement available relative to Alice has the same outcome probabilities in the two cases:
$$
P(y|q=0)=P(y|q=1).
$$
Bayes' rule therefore leaves her posterior equal to her prior.

The trace distances make the contrast explicit:
$$
D(\rho_0^A,\rho_1^A)=0,\qquad
D(\rho_0^B,\rho_1^B)=1.
$$
For priors $(p_0,p_1)$, the Helstrom success probability is$$
$$P_{\rm succ}=\frac12\left(1+
\|p_0\rho_0-p_1\rho_1\|_1\right).
$$

It is $\max(p_0,p_1)$ for Alice and 1 for Bob. The entropy of Bob's mixture is $H(p_0)$, since his two states are orthogonal.

This section mainly puts the result of Eq. (31) in the language of state discrimination. In this particular example it does not add a new distinction: Alice's relative states are identical by construction.

In [10]:
def tn(A): return np.sum(np.linalg.svd(A, compute_uv=False))
def outer(v): return np.outer(v, v.conj())
rA0, rA1 = outer(psiA0), outer(psiA1); rB0, rB1 = outer(psiB0), outer(psiB1)
print("trace distance, Alice:", 0.5*tn(rA0-rA1), "| Bob:", 0.5*tn(rB0-rB1))

# 1. Alice's posterior stays at her prior for any observable relative to A
Tr_ = rand_herm(4); vals, vecs = np.linalg.eigh(rel_obs_A(Tr_))
def P_y_given(psi): return np.array([abs(np.vdot(vecs[:,i], psi))**2 for i in range(8)])
print("P(y|q=0) == P(y|q=1) for all outcomes of a random relative observable:", np.allclose(P_y_given(psi0), P_y_given(psi1)))
# contrast: the charge observable X(x)X(x)X separates them (extra-particle information)
Gchg = kron(X, X, X)                                   # (G was reused as a matrix name in Section 7)
p_plus = lambda psi: np.vdot(psi, (np.eye(8)+Gchg)/2 @ psi).real
print("P(charge=+1): psi0 =", round(p_plus(psi0),3), "| psi1 =", round(p_plus(psi1),3), "-> posterior becomes certain once charge is accessible")

# 2-4. priors, Helstrom success, entropy
for p0 in (0.5, 0.8):
    p1 = 1-p0
    helA = 0.5*(1+tn(p0*rA0-p1*rA1)); helB = 0.5*(1+tn(p0*rB0-p1*rB1))
    ev = np.linalg.eigvalsh(p0*rB0+p1*rB1); ev = ev[ev>1e-12]; S = -np.sum(ev*np.log2(ev))
    print(f"p0={p0}: Helstrom success  Alice {helA:.3f} (prior guess {max(p0,p1):.3f}) | Bob {helB:.3f} | S(rho_B) = {S:.3f} bits")

trace distance, Alice: 0.0 | Bob: 1.0000000000000002
P(y|q=0) == P(y|q=1) for all outcomes of a random relative observable: True
P(charge=+1): psi0 = 1.0 | psi1 = 0.0 -> posterior becomes certain once charge is accessible
p0=0.5: Helstrom success  Alice 0.500 (prior guess 0.500) | Bob 1.000 | S(rho_B) = 1.000 bits
p0=0.8: Helstrom success  Alice 0.800 (prior guess 0.800) | Bob 1.000 | S(rho_B) = 0.722 bits


## 9. Can a non-ideal frame resolve different sectors?

The \(U(1)\) example above suggests a distinction between measurements generated by a single relational POVM and measurements in the full algebra generated by those operators. I tested whether states from different $(\kappa)$-sectors can be distinguished in these two settings.

For a global state $(\rho)$, define the dual map $(R_F^*)$ by
$$
\operatorname{tr}[\rho R_F(m)]
=\operatorname{tr}[R_F^*(\rho)m].
$$
Optimising over POVMs on the system gives
$$
D_{\rm single}(\rho,\rho')
=\frac12\|R_F^*(\rho)-R_F^*(\rho')\|_1.
$$
I sampled Haar-random pure states within fixed total-charge subspaces and compared pairs belonging to different $(\kappa)$-sectors. The reported averages are over 100 draws for each pair, so they depend on the sampled states.

The full generated algebra contains the projectors onto the $(\kappa)$-sectors in the examples tested. Those projectors distinguish states supported on different sectors. The numerical question is whether the same projectors can be obtained from the span of single relational operators.

In [11]:
def probe(sF, sS, K, nsamp=100, seed=7):
    global thetas, wq
    thetas = 2*np.pi*np.arange(K)/K; wq = 2*np.pi/K
    rg = np.random.default_rng(seed)
    ops = relational_ops(sF, sS); dS = len(sS); dF = len(sF)
    def basis_of(mats, tol=1e-9):
        M = np.array([m.flatten() for m in mats]); _,s_,vh = np.linalg.svd(M, full_matrices=False)
        return [vh[i].reshape(mats[0].shape) for i in range((s_>tol).sum())]
    B = basis_of(ops+[np.eye(ops[0].shape[0])])
    while True:
        new = basis_of(B+[a@b for a in B for b in B]+[a.conj().T for a in B])
        if len(new)==len(B): break
        B = new
    def in_span(Bs, X):
        M = np.array([b.flatten() for b in Bs]).T; c = np.linalg.lstsq(M, X.flatten(), rcond=None)[0]
        return np.linalg.norm(M@c - X.flatten()) < 1e-8
    ch = (np.kron(np.diag(sF),np.eye(dS))+np.kron(np.eye(dF),np.diag(sS))).diagonal().real
    rs = sorted(set(ch)); Pi = {r: np.diag(np.isclose(ch,r).astype(float)) for r in rs}
    kap = {r: frozenset(set(sS) & {r-a for a in sF}) for r in rs}
    sec = {}
    for r,k in kap.items(): sec.setdefault(k,[]).append(r)
    SP = {k: sum(Pi[r] for r in rr) for k,rr in sec.items()}
    alg_ok = bool(all(in_span(B,P) for P in SP.values()))
    single_flags = {tuple(float(x) for x in sorted(k)): bool(in_span(ops,P)) for k,P in SP.items()}
    def Rstar(rho):
        S_ = np.zeros((dS,dS),dtype=complex)
        for i in range(dS):
            for j in range(dS): S_[j,i] = np.trace(rho@ops[i*dS+j])
        return S_
    def rstate(r):
        idx = np.where(np.isclose(ch,r))[0]; v = np.zeros(len(ch),dtype=complex)
        v[idx] = rg.normal(size=len(idx))+1j*rg.normal(size=len(idx)); v/=np.linalg.norm(v); return np.outer(v,v.conj())
    means = []
    for r1 in rs:
        for r2 in rs:
            if r1<r2 and kap[r1]!=kap[r2]:
                means.append(np.mean([0.5*tn(Rstar(rstate(r1))-Rstar(rstate(r2))) for _ in range(nsamp)]))
    return alg_ok, single_flags, np.array(means)

for name,sF,sS,K in [("frame {-1,0,1}, system {-1,1}  (Ex. 5, A)", [-1,0,1], [-1,1], 32),
                     ("frame {-1,1}, system {-1,0,1}  (Ex. 5, B)", [-1,1], [-1,0,1], 32),
                     ("frame {-3..3}, system {-1,1}", list(range(-3,4)), [-1,1], 64)]:
    alg_ok, single_flags, means = probe(sF, sS, K)
    print(name)
    print("   sector projectors all in generated algebra:", alg_ok)
    print("   sector projector in span of single relational operators, per sector:", single_flags)
    print(f"   D_single over cross-sector pairs: mean of means {means.mean():.3f}, min {means.min():.3f}, share of pairs with mean>0.99: {(means>0.99).mean():.2f}")

frame {-1,0,1}, system {-1,1}  (Ex. 5, A)
   sector projectors all in generated algebra: True
   sector projector in span of single relational operators, per sector: {(-1.0,): False, (-1.0, 1.0): False, (1.0,): False}
   D_single over cross-sector pairs: mean of means 0.832, min 0.649, share of pairs with mean>0.99: 0.50
frame {-1,1}, system {-1,0,1}  (Ex. 5, B)
   sector projectors all in generated algebra: True
   sector projector in span of single relational operators, per sector: {(-1.0,): False, (0.0,): True, (-1.0, 1.0): False, (1.0,): False}
   D_single over cross-sector pairs: mean of means 0.926, min 0.655, share of pairs with mean>0.99: 0.78
frame {-3..3}, system {-1,1}
   sector projectors all in generated algebra: True
   sector projector in span of single relational operators, per sector: {(-1.0,): False, (-1.0, 1.0): False, (1.0,): False}
   D_single over cross-sector pairs: mean of means 0.715, min 0.619, share of pairs with mean>0.99: 0.17


**What I found**

- In all three examples, the sector projectors were present in the generated algebra. Thus the full algebra distinguishes states supported on different sectors.
- The single-operator span behaves differently in the two frame/system choices from the preprint. For frame A, none of the sector projectors tested lies in the span. For frame B, the projector for the sector \(\{0\}\) does lie in it.
- The random-state averages do not show a clean trend with frame size. The averages are about 0.83, 0.93 and 0.72 for the three cases, but the sets of sector pairs differ between cases.

So this numerical test does not support a simple statement that larger frames always make sectors harder to distinguish. It does suggest that the answer depends on which sectors are being compared and on the states within them. Section 11 replaces these averages with a worst-case and best-case analysis.

## 10. Sector distinguishability from the $(\kappa)$-sets

The averages in the previous section depend on the random states chosen. For pure states with definite total charge, the effective system state can be written down directly, which gives an exact criterion.

Take a global state of total charge $(r)$:
$$
|\Psi\rangle=\sum_{a+s=r}c_{as}|a\rangle_F|s\rangle_S.
$$
Contracting the frame with its orientation state gives
$$
(\langle\theta|_F\otimes I_S)|\Psi\rangle
=e^{-ir\theta}U_S(\theta)|\psi_S\rangle,
$$
where
$$
|\psi_S\rangle=\sum_{s\in\kappa(r)}
c_{r-s,s}|s\rangle_S,\qquad
\kappa(r)=\sigma_S\cap(r-\sigma_F).
$$
The vector $(|\psi_S\rangle)$ is normalised and lies in
$$
K(r)=\operatorname{span}\{|s\rangle:s\in\kappa(r)\}.
$$
Substituting this expression into the relationalisation map shows that
$$
\operatorname{tr}[\rho R_F(m)]
=\langle\psi_S|m|\psi_S\rangle.
$$
Thus, for a fixed total charge, a single relational measurement has the same statistics as a system measurement on an effective pure state in \(K(r)\).

For two such pure-state hypotheses, the optimal single-relational distinguishability is
$$
D=\sqrt{1-|\langle\psi_1|\psi_2\rangle|^2}.
$$

This gives the following consequences for two different sectors $(K(r_1))$ and $(K(r_2))$:

1. **Best case.** The maximum is \(D=1\), since different $(\kappa)$-sets contain at least one basis vector that is absent from the other set.
2. **Worst case.** If the two $(\kappa)$-sets overlap, both effective states can be chosen as the same basis vector, giving \(D=0\). If they are disjoint, every pair of effective states is orthogonal and \(D=1\).
3. **Full generated algebra.** The sector projectors distinguish states supported on different $(\kappa)$-sectors. They do not distinguish total-charge labels that have the same $(\kappa)$-set. Within a sector, other elements of the algebra may still distinguish different effective system states.
4. **Span observation.** In the examples below, a sector projector lies in the span of single relational operators exactly when its $(\kappa)$-set is disjoint from the other sector sets. I have checked this numerically for the four examples; I have not derived it generally.

The criterion for the single-relational measurement is therefore the intersection of the two $(\kappa)$-sets, rather than the size of the frame by itself.

In [12]:
def verify_criterion(sF, sS, K, nsamp=200, seed=3):
    global thetas, wq
    thetas = 2*np.pi*np.arange(K)/K; wq = 2*np.pi/K
    rg = np.random.default_rng(seed)
    ops = relational_ops(sF, sS); dS = len(sS); dF = len(sF)
    ch = (np.kron(np.diag(sF),np.eye(dS))+np.kron(np.eye(dF),np.diag(sS))).diagonal().real
    rs = sorted(set(ch)); kap = {r: frozenset(set(sS) & {r-a for a in sF}) for r in rs}
    def Rstar(rho):
        S_ = np.zeros((dS,dS),dtype=complex)
        for i in range(dS):
            for j in range(dS): S_[j,i] = np.trace(rho@ops[i*dS+j])
        return S_
    def vec_in(r, coeff=None):
        idx = np.where(np.isclose(ch,r))[0]; v = np.zeros(len(ch),dtype=complex)
        v[idx] = (rg.normal(size=len(idx))+1j*rg.normal(size=len(idx))) if coeff is None else coeff(idx); return v/np.linalg.norm(v)
    # (a) effective state is pure and supported on kappa(r)
    ok_a = True
    for r in rs:
        v = vec_in(r); S_ = Rstar(outer(v)); w = np.linalg.eigvalsh(S_)
        support = [j for j,sv in enumerate(sS) if sv in kap[r]]
        off = np.delete(np.delete(S_, support, 0), support, 1) if len(support)<dS else np.zeros((0,0))
        ok_a &= bool(abs(w[-1]-1)<1e-8 and abs(off).max(initial=0)<1e-8)
    # (b) worst/best case for cross-sector pairs
    ok_b = True; npairs = 0
    for r1 in rs:
        for r2 in rs:
            if r1<r2 and kap[r1]!=kap[r2]:
                npairs += 1
                Ds = [0.5*tn(Rstar(outer(vec_in(r1)))-Rstar(outer(vec_in(r2)))) for _ in range(nsamp)]
                if kap[r1] & kap[r2]:
                    s0 = sorted(kap[r1] & kap[r2])[0]                     # common s: product states |r-s>|s>
                    def basis_state(r, s0=s0):
                        a = r - s0; ia = sF.index(a); isx = sS.index(s0); v = np.zeros(len(ch),dtype=complex); v[ia*dS+isx] = 1; return v
                    d_worst = 0.5*tn(Rstar(outer(basis_state(r1)))-Rstar(outer(basis_state(r2))))
                    ok_b &= bool(d_worst<1e-9 and max(Ds)<=1+1e-9)
                else:
                    ok_b &= bool(min(Ds)>1-1e-9)                              # disjoint: perfect for every state
    # (c) sector projector in single span  <=>  kappa disjoint from all other sectors' kappa
    B1 = [o.flatten() for o in ops]; M = np.array(B1).T
    sec = {}
    for r,kk in kap.items(): sec.setdefault(kk,[]).append(r)
    ok_c = True
    for kk,rr in sec.items():
        P = np.diag(sum(np.isclose(ch,r).astype(float) for r in rr)).astype(complex)
        c = np.linalg.lstsq(M, P.flatten(), rcond=None)[0]; in_span = np.linalg.norm(M@c-P.flatten())<1e-8
        disjoint = all(not (kk & k2) for k2 in sec if k2!=kk)
        ok_c &= (bool(in_span)==disjoint)
    return ok_a, ok_b, ok_c, npairs

for name,sF,sS,K in [("frame {-1,0,1}, system {-1,1}", [-1,0,1], [-1,1], 32),
                     ("frame {-1,1}, system {-1,0,1}", [-1,1], [-1,0,1], 32),
                     ("frame {-3..3}, system {-1,1}", list(range(-3,4)), [-1,1], 64),
                     ("frame {-2,0,2}, system {-1,0,1,2}", [-2,0,2], [-1,0,1,2], 48)]:
    a,b,c,n = verify_criterion(sF,sS,K)
    print(f"{name}: effective state pure & supported on kappa(r): {a} | worst/best-case rule on {n} cross-sector pairs: {b} | span criterion: {c}")

frame {-1,0,1}, system {-1,1}: effective state pure & supported on kappa(r): True | worst/best-case rule on 8 cross-sector pairs: True | span criterion: True
frame {-1,1}, system {-1,0,1}: effective state pure & supported on kappa(r): True | worst/best-case rule on 9 cross-sector pairs: True | span criterion: True
frame {-3..3}, system {-1,1}: effective state pure & supported on kappa(r): True | worst/best-case rule on 24 cross-sector pairs: True | span criterion: True
frame {-2,0,2}, system {-1,0,1,2}: effective state pure & supported on kappa(r): True | worst/best-case rule on 26 cross-sector pairs: True | span criterion: True


The calculation above assumes pure global states with definite total charge and the rank-one, equal-weight orientation POVM used in this notebook. It does not cover mixtures of charge sectors or mixed hypotheses. The numerical checks test the effective-state construction and representative states attaining the extrema; the analytic argument, rather than a numerical search, establishes the extrema. The statement about when a sector projector lies in the single-operator span remains a numerical observation.

A natural next calculation would be to replace the pure-state hypotheses by mixed states with priors and use the Helstrom bound. I would also like to understand whether an analogous description is possible for non-Abelian groups.

## Concluding remarks

The three-qubit example illustrates that the same global states can look different from different quantum reference frames. Alice obtains identical relative states, while Bob obtains orthogonal ones. The global charge operator distinguishes the two states, but it is not an observable relative to Alice.

The later sections explore three directions: 
1. Extending the construction to $(Z_N)$ 
2. Reproducing a non-ideal $U(1)$-frame example 
3. Studying how the $\kappa$-sets control distinguishability under single relational measurements.

The last of these gives the cleanest mathematical question to take further. Its present scope is limited to pure states of definite charge, and its relation to existing work still needs to be checked.